# Notebook 05 — Entangling Gate Benchmark

This notebook extends the Rydberg model to a simple **gate-level benchmark**.

Rather than only tracking blockade or Bell-state fidelity, we now compare the evolution of the computational basis states:

$$
|gg\rangle,\ |gr\rangle,\ |rg\rangle,\ |rr\rangle
$$

and evaluate how well the dynamics approximates an **entangling operation** inspired by CZ-style behavior.

This notebook is **not** a full pulse-engineered gate design. Instead, it is a compact benchmark that asks:

- how distinguishable are the four basis-state responses,
- how does the interaction $V$ affect two-body phase-sensitive dynamics,
- how do decay and dephasing reduce gate quality?

We use a phase-sensitive **state-transfer benchmark** as a simple proxy for gate performance.


In [ ]:
# Ensure dependencies are installed before imports
import importlib, subprocess, sys

def ensure(pkg):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

for pkg in ['qutip', 'numpy', 'scipy', 'matplotlib']:
    ensure(pkg)


## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qutip import basis, qeye, tensor, sigmax, sesolve, mesolve, fidelity

## Basis states and operators

In [ ]:
g = basis(2, 0)
r = basis(2, 1)
I = qeye(2)

sx = sigmax()
n = r * r.dag()
sigma_minus = g * r.dag()

gg = tensor(g, g)
gr = tensor(g, r)
rg = tensor(r, g)
rr = tensor(r, r)

basis_states = {
    '|gg>': gg,
    '|gr>': gr,
    '|rg>': rg,
    '|rr>': rr,
}

sx1 = tensor(sx, I)
sx2 = tensor(I, sx)
n1 = tensor(n, I)
n2 = tensor(I, n)
sm1 = tensor(sigma_minus, I)
sm2 = tensor(I, sigma_minus)


## Hamiltonian and collapse operators

In [ ]:
def two_atom_hamiltonian(omega: float, delta: float, V: float):
    drive = 0.5 * omega * (sx1 + sx2)
    detuning = -delta * (n1 + n2)
    interaction = V * n1 * n2
    return drive + detuning + interaction


def collapse_operators(gamma_decay: float = 0.0, gamma_phi: float = 0.0):
    c_ops = []
    if gamma_decay > 0:
        c_ops.append(np.sqrt(gamma_decay) * sm1)
        c_ops.append(np.sqrt(gamma_decay) * sm2)
    if gamma_phi > 0:
        c_ops.append(np.sqrt(gamma_phi) * n1)
        c_ops.append(np.sqrt(gamma_phi) * n2)
    return c_ops

## Entangling benchmark metric

For an ideal CZ gate, the computational basis populations are preserved, while the $|11\rangle$ state acquires a minus sign.

In this toy encoding, we use $|g\rangle \leftrightarrow |0\rangle$ and $|r\rangle \leftrightarrow |1\rangle$.

We define a simple **state-by-state benchmark**:

- evolve each basis input state,
- compare the output against the corresponding ideal CZ-inspired target,
- report the average score and the worst-case score.

This is a useful first proxy for gate quality, even though it is **not** a full process tomography or exact CZ implementation.


In [ ]:
ideal_targets = {
    '|gg>': gg,
    '|gr>': gr,
    '|rg>': rg,
    '|rr>': -rr,
}


def evolve_state(psi0, omega: float, delta: float, V: float,
                 gamma_decay: float = 0.0,
                 gamma_phi: float = 0.0,
                 t_final: float = 4.0,
                 n_steps: int = 200):
    H = two_atom_hamiltonian(omega, delta, V)
    c_ops = collapse_operators(gamma_decay=gamma_decay, gamma_phi=gamma_phi)
    times = np.linspace(0.0, t_final, n_steps)

    if len(c_ops) == 0:
        result = sesolve(H, psi0, times)
        final_state = result.states[-1]
    else:
        result = mesolve(H, psi0, times, c_ops)
        final_state = result.states[-1]

    return times, result.states, final_state


def state_overlap_score(final_state, target_state):
    """Phase-sensitive state overlap score |<target|psi>|^2 or <target|rho|target>."""
    if final_state.isket:
        amp = target_state.overlap(final_state)
        return float(np.abs(amp) ** 2)
    else:
        val = (target_state.dag() * final_state * target_state)
        if hasattr(val, 'full'):
            return float(np.real(val.full()[0, 0]))
        return float(np.real(val))


def entangling_benchmark(omega: float, delta: float, V: float,
                         gamma_decay: float = 0.0,
                         gamma_phi: float = 0.0,
                         t_final: float = 4.0,
                         n_steps: int = 200):
    scores = {}
    for label, psi0 in basis_states.items():
        _, _, final_state = evolve_state(
            psi0,
            omega=omega,
            delta=delta,
            V=V,
            gamma_decay=gamma_decay,
            gamma_phi=gamma_phi,
            t_final=t_final,
            n_steps=n_steps,
        )
        scores[label] = state_overlap_score(final_state, ideal_targets[label])

    avg_score = float(np.mean(list(scores.values())))
    worst_score = float(np.min(list(scores.values())))
    return avg_score, worst_score, scores


## Basis-state benchmark at one operating point

In [ ]:
omega = 1.0
delta = 0.0
V = 4.0
t_gate = 4.0

avg_score, worst_score, scores = entangling_benchmark(omega=omega, delta=delta, V=V, t_final=t_gate)
print(f'Average entangling benchmark score: {avg_score:.4f}')
print(f'Worst-case basis-state score: {worst_score:.4f}')
for label, score in scores.items():
    print(f'{label}: {score:.4f}')

## Final-state scores by basis input

In [ ]:
labels = list(scores.keys())
values = [scores[k] for k in labels]

plt.figure(figsize=(7, 4.5))
plt.bar(labels, values)
plt.ylim(0, 1.05)
plt.ylabel('State-overlap score to ideal target')
plt.title('Basis-state entangling benchmark at a fixed operating point')
plt.tight_layout()
plt.show()

## Average entangling benchmark over $(\Omega, V)$ at fixed gate time

In [ ]:
omega_vals = np.linspace(0.2, 2.5, 32)
V_vals = np.linspace(0.0, 8.0, 32)

benchmark_landscape = np.zeros((len(V_vals), len(omega_vals)))
worst_landscape = np.zeros((len(V_vals), len(omega_vals)))

for i, V in enumerate(V_vals):
    for j, omega in enumerate(omega_vals):
        avg_score, worst_score, _ = entangling_benchmark(
            omega=omega,
            delta=0.0,
            V=V,
            gamma_decay=0.0,
            gamma_phi=0.0,
            t_final=4.0,
            n_steps=160,
        )
        benchmark_landscape[i, j] = avg_score
        worst_landscape[i, j] = worst_score


In [ ]:
plt.figure(figsize=(7.5, 5.5))
im = plt.imshow(
    benchmark_landscape,
    origin='lower',
    aspect='auto',
    extent=[omega_vals[0], omega_vals[-1], V_vals[0], V_vals[-1]],
)
plt.contour(omega_vals, V_vals, benchmark_landscape, colors='white', linewidths=0.4)
plt.xlabel(r'Rabi frequency $\Omega$')
plt.ylabel(r'Interaction strength $V$')
plt.title(r'Average entangling benchmark over $(\Omega, V)$')
plt.colorbar(im, label='average score')
plt.tight_layout()
plt.show()

plt.figure(figsize=(7.5, 5.5))
im = plt.imshow(
    worst_landscape,
    origin='lower',
    aspect='auto',
    extent=[omega_vals[0], omega_vals[-1], V_vals[0], V_vals[-1]],
)
plt.contour(omega_vals, V_vals, worst_landscape, colors='white', linewidths=0.4)
plt.xlabel(r'Rabi frequency $\Omega$')
plt.ylabel(r'Interaction strength $V$')
plt.title(r'Worst-case basis-state score over $(\Omega, V)$')
plt.colorbar(im, label='worst-case score')
plt.tight_layout()
plt.show()

## Noise sensitivity of the entangling benchmark

In [ ]:
gamma_decay_grid = np.linspace(0.0, 0.15, 26)
gamma_phi_grid = np.linspace(0.0, 0.15, 26)

noise_landscape = np.zeros((len(gamma_phi_grid), len(gamma_decay_grid)))

for i, gamma_phi in enumerate(gamma_phi_grid):
    for j, gamma_decay in enumerate(gamma_decay_grid):
        avg_score, _, _ = entangling_benchmark(
            omega=1.0,
            delta=0.0,
            V=4.0,
            gamma_decay=gamma_decay,
            gamma_phi=gamma_phi,
            t_final=4.0,
            n_steps=160,
        )
        noise_landscape[i, j] = avg_score

In [ ]:
plt.figure(figsize=(7.5, 5.5))
im = plt.imshow(
    noise_landscape,
    origin='lower',
    aspect='auto',
    extent=[gamma_decay_grid[0], gamma_decay_grid[-1], gamma_phi_grid[0], gamma_phi_grid[-1]],
)
plt.contour(gamma_decay_grid, gamma_phi_grid, noise_landscape, colors='white', linewidths=0.4)
plt.xlabel(r'Spontaneous emission $\gamma$')
plt.ylabel(r'Pure dephasing $\gamma_\phi$')
plt.title(r'Average entangling benchmark over $(\gamma, \gamma_\phi)$')
plt.colorbar(im, label='average score')
plt.tight_layout()
plt.show()

## Fixed-gate-time scan at one operating point

In [ ]:
gate_times = np.linspace(0.5, 6.0, 45)
avg_scores_vs_time = []
worst_scores_vs_time = []

for t_gate in gate_times:
    avg_score, worst_score, _ = entangling_benchmark(
        omega=1.0,
        delta=0.0,
        V=4.0,
        gamma_decay=0.0,
        gamma_phi=0.0,
        t_final=t_gate,
        n_steps=180,
    )
    avg_scores_vs_time.append(avg_score)
    worst_scores_vs_time.append(worst_score)


In [ ]:
plt.figure(figsize=(7, 4.5))
plt.plot(gate_times, avg_scores_vs_time, label='average score')
plt.plot(gate_times, worst_scores_vs_time, label='worst-case score')
plt.xlabel('Gate time')
plt.ylabel('Benchmark score')
plt.title('Fixed-gate-time entangling benchmark at a chosen operating point')
plt.legend()
plt.tight_layout()
plt.show()

## Best point in the coherent $(\Omega, V)$ scan

In [ ]:
best_index = np.unravel_index(np.argmax(benchmark_landscape), benchmark_landscape.shape)
best_V = V_vals[best_index[0]]
best_omega = omega_vals[best_index[1]]
best_avg = benchmark_landscape[best_index]
best_worst = worst_landscape[best_index]

print(f'Best coherent scan point: Omega = {best_omega:.3f}, V = {best_V:.3f}, average score = {best_avg:.4f}, worst-case score = {best_worst:.4f}')

## Interpretation

This notebook adds a gate-oriented view of the same physical model:

- basis-state scores show whether the dynamics approximately preserves computational-basis structure,
- the $|rr\rangle$ channel is the most sensitive to interaction-induced phase effects,
- the $(\Omega, V)$ landscape identifies regions where an entangling benchmark looks strongest,
- the worst-case score reveals when average performance hides a weak channel,
- the noise landscape shows how decay and dephasing reduce average gate quality.

This notebook demonstrates **entangling dynamics**, but it does **not** implement a full CZ gate because single-excitation channels are mixed and no pulse-engineered controlled phase is isolated.


## Optional next steps

Natural follow-ups are:

- replace this entangling benchmark with a full process-fidelity calculation,
- add pulse shaping $\Omega(t)$ and detuning ramps $\Delta(t)$,
- include distance-dependent interactions $V(R)$,
- compare this toy benchmark against a more realistic neutral-atom gate sequence.
